In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
path = "/Volumes/workspace/default/volume_1/employee.csv"
schema = StructType([
    StructField("empid",IntegerType(),True),
    StructField("empname",StringType(),True),
    StructField("address",StringType(),True),
    StructField("corrupted_records",StringType(),True)
    ])
df = spark.read.format("csv").option("header","true").option("inferschema","true").option("mode","PERMISSIVE").schema(schema).load(path)
df.show()

In [0]:
from pyspark.sql.functions import count,col
data = [(48, "Airport"), (49, "Office"),(50, "Hospital"),(51, "Airport"),(52, "Hospital"),(53, "Shoppingmall"),(54, "Office"),(55, "Hospital"),(56, "Hospital")]
schema = StructType([
    StructField("reqid", IntegerType(), True),
    StructField("pickup_location", StringType(), True)
])
pickup_df = spark.createDataFrame(data, schema)
pickup_df.groupBy("pickup_location").agg(count("pickup_location").alias("count")) \
                  .orderBy(col("count").desc()).limit(3) \
                  .select("pickup_location") \
                  .show()

In [0]:
from pyspark.sql.functions import count,col
from pyspark.sql.window import Window
data = [
    (100, 'IT', 100, '2024-05-12'),
    (200, 'IT', 100, '2024-06-12'),
    (100, 'FIN', 400, '2024-07-12'),
    (300, 'FIN', 500, '2024-07-12'),
    (300, 'FIN', 1543, '2024-07-12'),
    (300, 'FIN', 1500, '2024-07-12')
]
columns = ["empid", "dept", "salary", "date"]
df = spark.createDataFrame(data, columns)
window_con = Window.partitionBy("empid").orderBy()
result = df.withColumn("count",count("empid").over(window_con))
result.filter(col("count") == 1).select("empid", "dept", "salary", "date").show()

In [0]:
from pyspark.sql.functions import count,col,expr
data = [(72657),(1234),("Tom"),(8792),("Sam"),(19998),("Phillip")]
column = ["Empid"]
df = spark.createDataFrame(data, column)
result = (
    df.withColumn(
        "new_empid",
        expr("try_cast(Empid as int)")
    )
    .filter(col("new_empid").isNotNull())
)

result.show()

In [0]:
from pyspark.sql.functions import col,initcap
data = [("virat kohli",), ("p v sindhu",)]
schema = ["name"]
df = spark.createDataFrame(data, schema)
result=df.withColumn("name",initcap(col("name")))
result.show()

In [0]:
data1 = [(1, 'Bob'), (2, 'Alice'), (3, 'Tom')]
data2 = [(1, 'Bob'), (3, 'Tom')]
df1 = spark.createDataFrame(data1, ["id", "name"])
df2 = spark.createDataFrame(data2, ["id", "name"])
result = df1.join(df2, on="id", how="leftanti")
result.show()
#2nd method
result_1 =df1.subtract(df2).show()


In [0]:

from pyspark.sql.functions import col,initcap,cast,sum
data = [(1, None, 'ab'),
    (2, 10, None),
    (None, None, 'cd')]
columns = ['col1', 'col2', 'col3']
df = spark.createDataFrame(data, columns)
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()
         

In [0]:
data = [(101, 'IT', 1000), (102, 'HR', 900)]
columns = ["empid", "dept", "salary"]
df = spark.createDataFrame(data, columns)
prefix = "pref_"
result = df.select([col(c).alias(prefix+c) for c in df.columns])
result.show()

Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), 
        ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
#Solution
window_condition = Window.partitionBy("product_id").orderBy(col("date").desc())
result = df.withColumn("row_number_result",row_number().over(window_condition))
result.filter(col("row_number_result") == 1).select("product_id", "date", "sales").show()
       

2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data1 = [
    (101, "Narendra"),
    (102, "Ravi"),
    (103, "Anita")
]

columns1 = ["emp_id", "emp_name"]

df1 = spark.createDataFrame(data1, columns1)

data2 = [
    (104, "Suresh", 50000),
    (105, "Priya", 60000),
    (106, "Vijay", 70000)
]

columns2 = ["emp_id", "emp_name", "salary"]

df2 = spark.createDataFrame(data2, columns2)
#Writing dfs to parquet
path = "/Volumes/workspace/default/volume_for_parquet/employees"

# Write the first DataFrame
df1.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(path)

print("df1 written successfully")

# Append the second DataFrame
df2.write \
    .mode("append") \
    .format("parquet") \
    .save(path)

print("df2 written successfully")



In [0]:
#Read and merge parquet files.
df = (
    spark.read
         .option("mergeSchema", "true")
         .parquet("/Volumes/workspace/default/volume_for_parquet/employees")
)

df.printSchema()
df.show()

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?**

In [0]:
from pyspark.sql.functions import *
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df = df.groupBy("user_id").agg(sum("actions").alias("total_actions")).orderBy(col("total_actions").desc()).limit(5)
df.show()

**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
from pyspark.sql.window import Window
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
window_condition = Window.partitionBy("customer_id").orderBy(col("transaction_date").desc())
df = df.withColumn("most_recent_transaction",row_number().over(window_condition))
df.filter(col("most_recent_transaction") == 1).select("customer_id", "transaction_date", "sales").show()

**_7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?_**

In [0]:
from pyspark.sql.types import DateType
data = [("cust1", "2026-06-12"), ("cust2", "2026-04-20"), ("cust3", "2026-05-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("last_purchase_date",col("last_purchase_date").cast(DateType()))
df = df.withColumn("difference",datediff(current_date(),col("last_purchase_date"))).filter(col("difference")>30).show()

**_8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?_**

In [0]:

from pyspark.sql.functions import col,split,explode,count, regexp_replace
from pyspark.sql.window import Window
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("feedaback",regexp_replace(col("feedback"),"[^a-zA-Z0-9 ]","")) \
    .withColumn("word",explode(split(col("feedaback")," "))) 
df = df.groupBy("word").agg(count("word").alias("count")).orderBy(col("count").desc()).show()
    



**_9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?_**

In [0]:
from pyspark.sql.functions import col,sum
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("date",col("date").cast(DateType()))
window_condition = Window.partitionBy("product_id").orderBy(col("product_id").asc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)
df = df.withColumn("cumulative_sum",sum(col("sales")).over(window_condition)).show()

**_10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?_**


In [0]:
from pyspark.sql.functions import col,sum,row_number,lit
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df = df.dropDuplicates(["name", "age"]).show()


**_11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?_**

In [0]:
from pyspark.sql.types import DateType
from pyspark.sql.functions import col,avg
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df = df.groupBy("user_id").agg(avg(col("duration")).alias("avg_duration")).show()

**_12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this_**?

In [0]:
from pyspark.sql.functions import col,to_date,year,month,rank
from pyspark.sql.window import Window
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("date",to_date(col("date"))) \
    .withColumn("year",year(col("date"))) \
    .withColumn("month",month(col("date")))
window_con = Window.partitionBy("year","month").orderBy(col("sales").desc())
df = df.withColumn("rank",rank().over(window_con)).filter(col("rank") == 1).select("product_id", "date", "sales").show()


**_13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?_**

**_14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?_**

**_15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?_**

**16. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,count,rank
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]
df = spark.createDataFrame(data, columns)
df = df.groupBy("department").agg(count("employee_name").alias("no_of_emps")).orderBy(col("no_of_emps").desc()) 
df = df.withColumn("rank_value",rank().over(Window.orderBy(col("no_of_emps").desc()))).filter(col("rank_value")==1).select("department","no_of_emps").show()


**_- 23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition_**

In [0]:
from pyspark.sql.functions import col,when
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df = df.withColumn("sales_category", when(col("sales") >=200, "High")
                                    .otherwise("Low")).show()

_**24.While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**_

In [0]:
from pyspark.sql.functions import current_timestamp
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df = df.withColumn("processed_timestamp",current_timestamp()).show(truncate = False)

**_25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?_**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
sql_object = df.createOrReplaceTempView("sales_data")
result = spark.sql("SELECT * FROM sales_data")
result.show()

**_26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?_**

In [0]:
data = [("product1", 100),
        ("product2", 200),
        ("product3", 300)]

columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)

# Register as a Global Temporary View
df.createOrReplaceGlobalTempView("sales_data")

**_27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?_**

In [0]:
from pyspark.sql.functions import col

data = [
    ("product1", {"price": 100, "quantity": 2}),
    ("product2", {"price": 200, "quantity": 3})
]

columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)

df.printSchema()
#Since you created the DataFrame from Python dictionaries, product_info becomes a MapType, not a StructType.
flatten_df = (
    df.withColumn("price", col("product_info")["price"])
      .withColumn("quantity", col("product_info")["quantity"])
      .drop("product_info")
)

df2 = flatten_df.createOrReplaceTempView("product")
result = spark.sql("""
                       SELECT product_id, price, quantity
                       FROM product
                       """
)
result.show()

**_28. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?_**

In [0]:
✅ Use an explicit schema (StructType)
✅ Handle missing fields (NULL or default values)
✅ Validate data quality
✅ Separate bad records (quarantine/error table)
✅ Log errors instead of failing the job
✅ Support schema evolution (e.g., mergeSchema with Delta Lake)
✅ Continue processing valid records (build a robust pipeline)

**_While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?_**


In [0]:
"Partitioning is performed while writing the Parquet data using partitionBy(). When reading, Spark automatically leverages those partitions through partition pruning. For example, if the data is partitioned by country, filtering on country='India' causes Spark to read only the country=India partition instead of scanning all files, significantly improving query performance."

**_You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?_**

In [0]:
data = [
    ("product1", 100),
    ("product2", 200),
    ("product3", 300)
]

columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)

df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("/tmp/parquet_data")

**Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?**

In [0]:
1. Read only required columns.
2. Filter early (Predicate Pushdown)
3. Partition Pruning
If the Delta table is partitioned by date:
df = spark.read.format("delta").load("/mnt/delta/sales")
df.filter(col("date") == "2026-07-01")
4.OPTIMIZE
spark.sql("""
OPTIMIZE sales
""")
5. Z-ORDER
6. Cache intermediate result
7. Broadcast small tables
11. Use Delta Lake maintenance

**_You are processing sales data. Group by product categories and create a list of all product names in each category._**

In [0]:
from pyspark.sql.functions import concat_ws,col,collect_list
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.groupBy("category").agg(concat_ws(",", collect_list("product")).alias("products")).show(truncate=False)

**_You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased._**

In [0]:
from pyspark.sql.functions import col,collect_set
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
result = (
    df.groupBy("customer_id")
      .agg(collect_set("product_id").alias("products"))
)

result.show(truncate=False)

**_For customer records, combine first and last names only if the email address exists._**

In [0]:
from pyspark.sql.functions import concat_ws,col,when
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("full_name", when(col("email").isNotNull(),concat_ws(" ",col("first_name"),col("last_name")))
              .otherwise(None)
              ).show()

**_You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased._**

In [0]:
from pyspark.sql.functions import size,col
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df = df.withColumn("count_of_products",size(col("product_ids")))
df.show(truncate = False)

**_You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes._**

In [0]:
from pyspark.sql.functions import col,lpad
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df = df.withColumn("new_id",lpad(col("employee_id"),6,"0")).show()
              

_**You need to validate phone numbers by checking if they start with "91"**_

In [0]:

from pyspark.sql.functions import col
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
df = df.withColumn("is_valid", col("phone_number").startswith("91")).show()
                   

**_You have a dataset with courses taken by students. Calculate the average number of courses per student._**

In [0]:
from pyspark.sql.functions import col,avg,size
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
result = (
    df.withColumn("course_count", size("courses")) \
        .agg(avg(col("course_count")).alias("avg_courses")) 
)

result.show()

**_You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number._**

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df = df.withColumn("Main_Concact_number",when(col("primary_contact").isNull(),col("secondary_contact")) 
                                    .otherwise(col("primary_contact"))
              ).show()

_** You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**_

In [0]:
from pyspark.sql.functions import length,when,col
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df = df.withColumn("product", when (length(col("product_code")) ==5 ,"product_code")
                      .otherwise("Custom")
             ).select("product_code","product").show()

**_RDD-Resilient distributed Dataset_**

**_1. What is an RDD?_**

RDD (Resilient Distributed Dataset) is the fundamental data structure in Apache Spark.

It is:

Resilient → Fault-tolerant (can recover lost data using lineage)
Distributed → Data is divided into partitions and stored across multiple machines (executors)
Dataset → A collection of data

**_Interview Definition:_**

"RDD is an immutable, distributed collection of objects that can be processed in parallel across a cluster. It is fault-tolerant because Spark maintains lineage information to recompute lost partitions."

**_2. Why was RDD introduced?_**

Before Spark, Hadoop MapReduce processed data like this:

Read from Disk
      ↓
Map
      ↓
Write to Disk
      ↓
Read Again
      ↓
Reduce
      ↓
Write Again

Every intermediate result was written to disk.

Problems:

Slow
High disk I/O
Poor performance for iterative algorithms

Spark introduced RDDs to keep intermediate data in memory whenever possible.

**_3. Why do we need RDD?_**

RDDs solve several problems:
**_
Parallel Processing_**

Instead of one machine processing 1 TB of data:

Machine 1
1 TB

Spark distributes the data:

Machine 1 → 250 GB

Machine 2 → 250 GB

Machine 3 → 250 GB

Machine 4 → 250 GB

Each executor processes its partition in parallel.

**_Fault Tolerance_**

Suppose one executor crashes.

Executor 1

Partition 1

Spark doesn't restart the whole job.

It uses lineage to recreate only the lost partition.

**_Scalability_**

Need more processing power?

Just add more worker nodes.

4 Workers

↓

20 Workers

Spark automatically distributes partitions.

**_In-memory Computation_**

RDDs can be cached.

RDD

↓

cache()

↓

Memory

Subsequent operations become much faster.

**_SparkContext is the entry point to Spark Core._**

**It is responsible for:**
- Creating RDDs
- Connecting to the cluster
- Sending jobs to executors
- Managing resources
- Scheduling tasks

In [0]:
#Steps to create Rdd and see the data in it

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [("sunkagarla","Narendra"),("Sunkagarla","Salamma"),("Sunkagarla","Ramaiah")]
schema = ["First_name","Last_name"]
rdd = spark.sparkContext.parallelize(data,5)
rdd.collect()
